# OpenVLA 4-bit Quantization with SINQ

This script quantizes OpenVLA-7B to 4-bit using SINQ
and saves it to Google Drive for LIBERO benchmarking




In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Create directory for quantized model
import os
quantized_model_path = "/content/drive/MyDrive/openvla_4bit_sinq"
os.makedirs(quantized_model_path, exist_ok=True)
print(f"Quantized model will be saved to: {quantized_model_path}")

Quantized model will be saved to: /content/drive/MyDrive/openvla_4bit_sinq


In [5]:

!pip install transformers==4.38.2
!pip install torch torchvision accelerate
!pip install git+https://github.com/huawei-csl/SINQ.git  # Install SINQ from source
!pip install safetensors
!pip install timm==0.9.16


  Cloning https://github.com/huawei-csl/SINQ.git to /tmp/pip-req-build-xzk1iybf
  Running command git clone --filter=blob:none --quiet https://github.com/huawei-csl/SINQ.git /tmp/pip-req-build-xzk1iybf
  Resolved https://github.com/huawei-csl/SINQ.git to commit 6930f78dfe1d5df0c4e3e6b8db8816c3ac21498a
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 46.0 MB/s eta 0:00:00
  Attempting uninstall: timm
    Found existing installation: timm 1.0.25
    Uninstalling timm-1.0.25:
      Successfully uninstalled timm-1.0.25


In [4]:


import torch
import warnings
warnings.filterwarnings('ignore')

# Load with compatible transformers version
from transformers import AutoTokenizer, AutoModelForVision2Seq


In [ ]:
# Using these to load openvla using automodelvision2seq
# !pip install transformers=="4.40.1" # Complete installation for OpenVLA quantization
# !pip install transformers==4.38.2
# !pip install torch>=2.0.0
# !pip install accelerate>=0.20.0
# !pip install bitsandbytes>=0.41.0
# !pip install einops  # Required for some OpenVLA operations
# !pip install timm=="0.9.16"


In [1]:

# import transformers
# import torch
# import bitsandbytes as bnb
import timm

# print(f"Transformers: {transformers.__version__}")
# print(f"PyTorch: {torch.__version__}")
print(f"BitsAndBytes: {timm.__version__}")

BitsAndBytes: 0.9.16


In [ ]:
# #Grab Layer data

# from transformers import AutoModelForVision2Seq

# model_name = "openvla/openvla-7b-finetuned-libero-spatial"

# model = AutoModelForVision2Seq.from_pretrained(
#     model_name,
#     torch_dtype="auto",
#     device_map="cpu",
#     trust_remote_code=True
# )


# for name, _ in model.named_modules():
#     # if "llama" in name.lower() or "model" in name:
#         print(name)

In [5]:
MODEL_NAME = "openvla/openvla-7b-finetuned-libero-spatial"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print("Loading model (this requires transformers 4.38.2)...")
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)


Loading tokenizer...
Loading model (this requires transformers 4.38.2)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

In [12]:
# Import SINQ's standalone quantization functions
from sinq.patch_model import AutoSINQHFModel
from sinq.sinqlinear import BaseQuantizeConfig

In [14]:
# custom_sinq.py
from sinq.patch_model import BaseSINQHFModel, BasePatch

class CustomSINQHFModel(BaseSINQHFModel, BasePatch):
    @classmethod
    def get_ignore_layers(cls, model):
        modules_to_exclude = [
            "lm_head",
            "embed_tokens",
            "vision_backbone",
            "projector"
        ]

        layers = {""}
        for name, module in model.named_modules():
            if len(module._modules) == 0:
                continue
            if any(skip_term in name for skip_term in modules_to_exclude):
                layers.add(name)
                continue
            has_direct_params = any(p is not None for p in module.parameters(recurse=False))
            has_direct_bufs = any(b is not None for b in module.buffers(recurse=False))
            if not (has_direct_params or has_direct_bufs):
                layers.add(name)
        return list(layers)

In [15]:
from custom_sinq import CustomSINQHFModel

qmodel = CustomSINQHFModel.quantize_model(
    model,
    tokenizer=tokenizer,
    quant_config=quant_cfg,
    compute_dtype=torch.bfloat16,
    device=model.device,
)

ModuleNotFoundError: No module named 'custom_sinq'

In [17]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sinq.patch_model import AutoSINQHFModel, BaseSINQHFModel, BasePatch
from sinq.sinqlinear import BaseQuantizeConfig

# Step 1: Create a custom class that overrides get_ignore_layers
class CustomSINQHFModel(BaseSINQHFModel, BasePatch):
    """Custom SINQ model with excluded modules"""

    @classmethod
    def get_ignore_layers(cls, model):
        """Override to exclude specific modules from quantization"""
        # Modules to exclude from quantization
        modules_to_exclude = [
            "lm_head",
            "embed_tokens",
            "vision_backbone",
            "projector"
        ]

        layers = {""}
        for name, module in model.named_modules():
            # Skip leaf modules
            if len(module._modules) == 0:
                continue

            # Skip if module name contains any excluded term
            if any(skip_term in name for skip_term in modules_to_exclude):
                layers.add(name)
                continue

            # Original logic: skip non-leaf modules without direct params
            has_direct_params = any(p is not None for p in module.parameters(recurse=False))
            has_direct_bufs = any(b is not None for b in module.buffers(recurse=False))
            if not (has_direct_params or has_direct_bufs):
                layers.add(name)

        return list(layers)


# Step 2: Create quantization config (dont need modules_to_exclude here)
quant_cfg = BaseQuantizeConfig(
    nbits=4,            # 4-bit quantization
    group_size=64,      # Group size for quantization
    tiling_mode="1D",   # Tiling strategy
    method="sinq",      # Quantization method
)

# Step 3: Use custom class for quantization

MODEL_NAME = "openvla/openvla-7b-finetuned-libero-spatial"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print("Loading model (this requires transformers 4.38.2)...")
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

qmodel = CustomSINQHFModel.quantize_model(  # <-- Use CustomSINQHFModel instead of AutoSINQHFModel
    model,
    tokenizer=tokenizer,
    quant_config=quant_cfg,
    compute_dtype=torch.bfloat16,
    device=model.device,
)

print("✓ Model quantized with SINQ (with excluded modules)")

Loading tokenizer...
Loading model (this requires transformers 4.38.2)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Default model structure not supported. Make sure you feed device as dictionary as {name_block: device}


TypeError: 'NoneType' object is not iterable

In [10]:
# ==============================================
#  Quantization Configuration
# ==============================================

# Create SINQ quantization config (standalone version)
quant_cfg = BaseQuantizeConfig(
    nbits=4,            # 4-bit quantization
    group_size=64,      # Group size for quantization
    tiling_mode="1D",   # Tiling strategy
    method="sinq",      # Quantization method
    # Specify modules to exclude from quantization
    modules_to_exclude = [
      "lm_head",
      "embed_tokens",
      "vision_backbone",
      "vision_backbone.model",
      "projector"]
)



TypeError: sinq_base_quant_config() got an unexpected keyword argument 'modules_to_exclude'

In [8]:
# Apply quantization
qmodel = AutoSINQHFModel.quantize_model(
    model,
    tokenizer=tokenizer,
    quant_config=quant_cfg,
    compute_dtype=torch.bfloat16,
    device=model.device,  # SINQ standalone should support this
)

print("✓ Model quantized with SINQ (standalone mode)")

TypeError: BaseSINQModel.quantize_model() got an unexpected keyword argument 'modules_to_not_convert'

In [ ]:
print("\nVerifying quantization...")

# Check memory reduction
def calculate_memory(model):
    total_params = sum(p.numel() for p in model.parameters())
    quantized_size = 0
    full_size = 0

    for name, param in model.named_parameters():
        if param.dtype == torch.uint8:
            quantized_size += param.numel() * 0.5  # 4-bit = 0.5 bytes per param
        else:
            full_size += param.numel() * 2  # bfloat16 = 2 bytes per param

    total_bytes = quantized_size + full_size
    return total_params, total_bytes / (1024**3)

total_params, memory_gb = calculate_memory(qmodel)
baseline_gb = total_params * 2 / (1024**3)

print(f"Total parameters: {total_params:,}")
print(f"Quantized memory: {memory_gb:.2f} GB")
print(f"Baseline (FP16): {baseline_gb:.2f} GB")
print(f"Memory reduction: {(1 - memory_gb / baseline_gb) * 100:.1f}%")


In [ ]:
# Save using SINQ's standalone save function
save_dir = "openvla-7b-libero-spatial-4bit-sinq"

print(f"\nSaving quantized model to {save_dir}...")
AutoSINQHFModel.save_quantized_safetensors(
    qmodel,
    tokenizer,
    save_dir,
    verbose=True,
    max_shard_size="5GB"
)

print("✓ Model saved successfully!")

In [8]:

# Configuration
MODEL_NAME = "openvla/openvla-7b-finetuned-libero-spatial"
QUANTIZATION_CONFIG = SinqConfig(
    nbits=4,                    # 4-bit quantization
    group_size=64,              # Group size for quantization
    tiling_mode="1D",         # Tiling strategy
    method="sinq",            # Quantization method
    modules_to_not_convert=[
    "lm_head",
    "embed_tokens",
    "vision_backbone",
    "vision_backbone.model",
    "projector"]
)

print("🎯 Starting OpenVLA 4-bit quantization with SINQ...")
print(f"Model: {MODEL_NAME}")
print(f"Quantization: {QUANTIZATION_CONFIG.nbits}-bit, group_size={QUANTIZATION_CONFIG.group_size}")

🎯 Starting OpenVLA 4-bit quantization with SINQ...
Model: openvla/openvla-7b-finetuned-libero-spatial
Quantization: 4-bit, group_size=64


In [16]:
# ==============================================
# Model Loading and Quantization
# ==============================================


# Load model with SINQ quantization
model = AutoModelVision2Seq.from_pretrained(
    MODEL_NAME,
    quantization_config=QUANTIZATION_CONFIG,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

print("✅ Model loaded with SINQ quantization!")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
print("✅ Tokenizer loaded!")




NameError: name 'AutoModelVision2Seq' is not defined

In [15]:
from transformers import AutoProcessor
from modeling_prismatic import PrismaticForConditionalGeneration
import torch

# Load Processor
processor = AutoProcessor.from_pretrained("openvla/openvla-7b", trust_remote_code=True)

# Load VLA with the specific model class
vla = PrismaticForConditionalGeneration.from_pretrained(
    "openvla/openvla-7b",
    attn_implementation="flash_attention_2",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

ModuleNotFoundError: No module named 'modeling_prismatic'

In [ ]:
# ==============================================
# Save Quantized Model
# ==============================================

print("saving quantized model to drive")

# Save the quantized model
model.save_pretrained(
    quantized_model_path,
    safe_serialization=True,
    max_shard_size="5GB"
)

# Save tokenizer
tokenizer.save_pretrained(quantized_model_path)

# Save quantization config
import json
config_path = os.path.join(quantized_model_path, "sinq_config.json")
with open(config_path, 'w') as f:
    json.dump({
        "nbits": QUANTIZATION_CONFIG.nbits,
        "group_size": QUANTIZATION_CONFIG.group_size,
        "tiling_mode": QUANTIZATION_CONFIG.tiling_mode,
        "method": QUANTIZATION_CONFIG.method,
        "modules_to_not_convert": QUANTIZATION_CONFIG.modules_to_not_convert
    }, f, indent=2)

print(f"Quantized model saved to: {quantized_model_path}")




In [ ]:
# ==============================================
# Memory Usage Statistics
# ==============================================

def get_model_size_mb(model):
    """Calculate model size in MB"""
    param_size = 0
    buffer_size = 0

    for param in model.parameters():
        param_size += param.nelement() * param.element_size()

    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()

    size_mb = (param_size + buffer_size) / 1024 / 1024
    return size_mb

original_size = 7000  # Approximate size of OpenVLA-7B in MB
quantized_size = get_model_size_mb(model)
compression_ratio = original_size / quantized_size

print(f"\n📈 Quantization Results:")
print(f"Original model size: ~{original_size} MB")
print(f"Quantized model size: {quantized_size:.1f} MB")
print(f"Compression ratio: {compression_ratio:.1f}x")
print(f"Memory reduction: {(1 - quantized_size/original_size)*100:.1f}%")